In [ ]:
import mne
import numpy as np
import os
import re

from mne_bids import BIDSPath, write_raw_bids

In [ ]:
# project directory
proj_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM2"


# raw data directory (need to specify for your computer)
data_dir = os.path.join(proj_dir, 'eeg_raw')
 
data_dir = os.path.join(proj_dir, 'eeg_raw')

In [ ]:
# output BIDS directory to be created/filled
bids_dir = os.path.join(proj_dir,
                        'data-bids')

In [ ]:
# event dicionary for stimulus name and code number
event_dict = {'base_pol1': 3,
              'base_pol2': 4,
              'high_pol1': 5,
              'high_pol2': 6}

In [ ]:
# read in pilot data for EAM2, just doing passive right now
sub = 'pilot1'
ses = '2'
# tasks = ['active', 'passive', 'motor']
tasks = ['passive']

pattern = re.compile(
    rf"sub-{sub}.*ses-{ses}.*({'|'.join(tasks)}).*acq-sched(\d+).*run-(\d+)"
)

# get all data paths
eeg_data_paths = os.listdir(data_dir)

for eeg_path in eeg_data_paths:
    match = pattern.search(eeg_path)
    if not match:
        continue

    task, sched, run = match.groups(1)


    # read raw eeg data
    raw_data = mne.io.read_raw_bdf(os.path.join(data_dir, eeg_path))

    # find events in Status
    events_raw = mne.find_events(raw_data, 
                                stim_channel='Status', 
                                initial_event=True)
    
    # drop start events
    # drop_idx = np.where(events_raw[:, 2] > max(event_dict.values()))
    # events_raw = np.delete(events_raw, drop_idx, axis=0)
    events = mne.pick_events(events_raw, include=list(event_dict.values()))
    
    # create BIDS path
    bids_path = BIDSPath(subject=sub,
                         session=ses,
                         task=task,
                         run=run,
                         acquisition=f'sched{sched}',
                         root=bids_dir)
    
    # write to BIDS
    write_raw_bids(raw_data, bids_path, events=events, event_id=event_dict, overwrite=True)



In [ ]:
# kevin's code
# define input EEG file
# bad participants: 01, 08, 13
for sub_num in range(2, 35):
    for task_label in ['active', 'passive']:
        for run_label in range(1,6):
            sub_label = f'{sub_num:02d}'

            data_fpath = os.path.join(data_dir,
                                    f'sub-{sub_label}_task-{task_label}_run-{run_label}.bdf')
            
            try:
                # read in input EEG file
                raw_data = mne.io.read_raw_bdf(data_fpath)

                # find events in the `Status` channel of the raw data
                events_raw = mne.find_events(raw_data, 
                                            stim_channel='Status', 
                                            initial_event=True)
                '''
                # explore the data
                events_raw[events_raw[:,2]==2049]
                '''
                '''
                # Modify event codes based on run type
                if int(sub_label) < 13:
                    if task_label=='active':
                        # change odd-indexed sound events to code 2050
                        odd_events = [] # TODO - array of odd indices
                        events_raw[events_raw[:,2]==2049][odd_events] = 2050
                '''        
                # run the function to generate events
                event_dict = generate_event_dict(sub_label, task_label)
                print('task events:', event_dict)

                # only keep task-related events (currently active and passive runs only)
                events = mne.pick_events(events_raw, include=[3, 4, 5, 6])
                print('task event codes in this run:', np.unique(events[:,2]))

                # define the BIDSPath object for converting this EEG file
                bids_path = BIDSPath(subject=str(sub_label), 
                                    run=run_label, 
                                    task=task_label,
                                    datatype='eeg', 
                                    root=bids_dir, )
                
                # finally, do the raw-to-BIDS conversion
                write_raw_bids(raw_data, 
                            bids_path=bids_path,   
                            events=events,
                            event_id=event_dict,
                            overwrite=True
                            )
            except FileNotFoundError:
                print('file does not exist:', data_fpath)
            except ValueError:
                print('need to fix events issue')

In [ ]:
(f'{sub_num:02d}')